In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.utils import add_self_loops, softmax

import warnings
warnings.filterwarnings('ignore')

/opt/anaconda3/envs/topicgraph_py310/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
class GATLayer(nn.Module):
    def __init__(self, in_channels, out_channels, dropout=0.6, alpha=0.2):
        super().__init__()

        self.W = nn.Linear(in_channels, out_channels, bias=False)
        self.a = nn.Parameter(torch.empty(2 * out_channels, 1))

        self.dropout = dropout
        self.alpha = alpha

        nn.init.xavier_uniform_(self.W.weight)
        nn.init.xavier_uniform_(self.a)

    def forward(self, x, edge_index, num_nodes):
            print("\n[0] 입력 x")
            print(x)
            print("shape:", x.shape)

            print("\n[1] 입력 edge_index")
            print(edge_index)
            print("shape:", edge_index.shape)

            edge_index, _ = add_self_loops(edge_index, num_nodes=num_nodes)
            print("\n[2] self-loop 추가 후 edge_index")
            print(edge_index)
            print("shape:", edge_index.shape)

            row, col = edge_index
            print("\n[3] row (source)")
            print(row)
            print("shape:", row.shape)

            print("\n[4] col (target)")
            print(col)
            print("shape:", col.shape)

            x = self.W(x)
            print("\n[5] 선형변환 후 x = W(x)")
            print(x)
            print("shape:", x.shape)

            x_i = x[col]   # target node feature
            x_j = x[row]   # source node feature

            print("\n[6] x_i = x[col] # 각 edge의 target 노드 feature")
            print(x_i)
            print("shape:", x_i.shape)

            print("\n[7] x_j = x[row] # 각 edge의 source 노드 feature")
            print(x_j)
            print("shape:", x_j.shape)

            cat_ij = torch.cat([x_i, x_j], dim=1)
            print("\n[8] concat([x_i, x_j])")
            print(cat_ij)
            print("shape:", cat_ij.shape)

            e = cat_ij @ self.a
            print("\n[9] e_before_activation = concat @ a")
            print(e)
            print("shape:", e.shape)

            e = F.leaky_relu(e.squeeze(-1), negative_slope=0.2)
            print("\n[10] e = LeakyReLU(...)")
            print(e)
            print("shape:", e.shape)

            alpha = softmax(e, col)
            print("\n[11] alpha = softmax(e, col)")
            print(alpha)
            print("shape:", alpha.shape)

            message = alpha.unsqueeze(-1) * x_j
            print("\n[12] message = alpha.unsqueeze(-1) * x_j")
            print(message)
            print("shape:", message.shape)

            out = torch.zeros(num_nodes, x.size(1), device=x.device)
            print("\n[13] 초기 out")
            print(out)
            print("shape:", out.shape)

            print("\n[14] edge별로 out[target] += message")
            for k in range(edge_index.size(1)):
                print(
                    f"edge {k}: {row[k].item()} -> {col[k].item()}, "
                    f"alpha={alpha[k].item():.4f}, "
                    f"message={message[k].tolist()}"
                )

            out.index_add_(0, col, message)

            print("\n[15] 최종 out")
            print(out)
            print("shape:", out.shape)

            return out

In [6]:
# -----------------------------
# 예시 데이터
# -----------------------------
x = torch.tensor([
    [1.0],   # node 0
    [2.0],   # node 1
    [3.0],   # node 2
])

# 무방향 그래프 0-1, 1-2
edge_index = torch.tensor([
    [0, 1, 1, 2],
    [1, 0, 2, 1]
])

num_nodes = 3

gat = GATLayer(in_channels=1, out_channels=1)

# 가중치 고정해서 디버깅 쉽게
with torch.no_grad():
    gat.W.weight[:] = torch.tensor([[2.0]])
    gat.a[:] = torch.tensor([
        [1.0],   # target 쪽 가중치
        [1.0],   # source 쪽 가중치
    ])

out = gat(x, edge_index, num_nodes)


[0] 입력 x
tensor([[1.],
        [2.],
        [3.]])
shape: torch.Size([3, 1])

[1] 입력 edge_index
tensor([[0, 1, 1, 2],
        [1, 0, 2, 1]])
shape: torch.Size([2, 4])

[2] self-loop 추가 후 edge_index
tensor([[0, 1, 1, 2, 0, 1, 2],
        [1, 0, 2, 1, 0, 1, 2]])
shape: torch.Size([2, 7])

[3] row (source)
tensor([0, 1, 1, 2, 0, 1, 2])
shape: torch.Size([7])

[4] col (target)
tensor([1, 0, 2, 1, 0, 1, 2])
shape: torch.Size([7])

[5] 선형변환 후 x = W(x)
tensor([[2.],
        [4.],
        [6.]], grad_fn=<MmBackward0>)
shape: torch.Size([3, 1])

[6] x_i = x[col] # 각 edge의 target 노드 feature
tensor([[4.],
        [2.],
        [6.],
        [4.],
        [2.],
        [4.],
        [6.]], grad_fn=<IndexBackward0>)
shape: torch.Size([7, 1])

[7] x_j = x[row] # 각 edge의 source 노드 feature
tensor([[2.],
        [4.],
        [4.],
        [6.],
        [2.],
        [4.],
        [6.]], grad_fn=<IndexBackward0>)
shape: torch.Size([7, 1])

[8] concat([x_i, x_j])
tensor([[4., 2.],
        [2., 4.],
  